# 📈 Riesgo y rendimiento de activos financieros
**Equipo:** Stattum · **Curso:** Lenguajes de Programación · Ingeniería Financiera · IX Concurso ITM 2026

**Pregunta principal:** ¿Cómo se comportaron el rendimiento y el riesgo de Almacenes Éxito, Ecopetrol y Grupo Cibest frente al mercado colombiano (ICOLCAP) durante los últimos cinco años, y cuál ofreció la mejor relación rendimiento-riesgo?

**Flujo:** parámetros → descarga de datos → retornos → indicadores → gráficos → interpretación.

> Los resultados históricos no garantizan rendimientos futuros. Este análisis es descriptivo, no una recomendación de inversión.

## 1. Instalación (solo la primera vez)

In [1]:
# Descomenta la siguiente línea si te faltan librerías (en Colab ya vienen casi todas)
# !pip install yfinance pandas numpy plotly

## 2. Parámetros
Cambia estos valores para modificar el análisis. Los símbolos `.CL` son de la Bolsa de Valores de Colombia: **confírmalos en Yahoo Finance** y revisa desde qué fecha tiene historial `PFCIBEST.CL` (Cibest cambió de ticker). Los precios están en pesos colombianos (COP).

In [2]:
from datetime import date, timedelta

TICKERS = ["EXITO.CL", "ECOPETROL.CL", "PFCIBEST.CL", "ICOLCAP.CL"]   # empresas colombianas (BVC, precios en COP)
BENCHMARK = "ICOLCAP.CL"                        # ETF que replica el COLCAP (para la beta)
INICIO = date(2021, 11, 9)                      # periodo fijo: los resultados no cambian entre ejecuciones
FIN = date(2026, 11, 6)
RF = 0.04                                       # tasa libre de riesgo anual: AJÚSTALA a una referencia colombiana (p. ej. TES) y cita la fuente
VENTANA = 60                                    # días para la volatilidad móvil
RELLENAR = True                                 # True: rellena huecos de hasta 5 días (acciones poco líquidas)

## 3. Descarga y preparación de datos
Se usa `yfinance` con precios ajustados (`auto_adjust=True`). La fecha final de `yf.download` es exclusiva, por eso se suma un día. Solo se conservan las **fechas comunes** a todos los activos.

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf

DIAS_ANIO = 252


def cargar_precios(tickers, inicio, fin):
    df = yf.download(tickers, start=str(inicio), end=str(fin + timedelta(days=1)),
                     auto_adjust=True, progress=False)["Close"]
    if isinstance(df, pd.Series):
        df = df.to_frame(tickers[0])
    return df


crudo = cargar_precios(TICKERS, INICIO, FIN)

sin_datos = [t for t in TICKERS if t not in crudo.columns or crudo[t].dropna().empty]
if sin_datos:
    print("⚠️ Sin datos para:", sin_datos, "→ se excluyen del análisis")

precios = crudo.drop(columns=sin_datos, errors="ignore")
if RELLENAR:
    precios = precios.ffill(limit=5)
precios = precios.dropna()          # fechas comunes a todos los activos

assert precios.shape[1] >= 2 and len(precios) >= 30, "Datos insuficientes: amplía el periodo o cambia los activos"
if BENCHMARK not in precios.columns:
    BENCHMARK = precios.columns[0]

print(f"{precios.shape[1]} activos · {len(precios)} días · {precios.index[0].date()} → {precios.index[-1].date()}")
precios.tail()

4 activos · 1288 días · 2021-11-09 → 2026-11-06


Ticker,ECOPETROL.CL,EXITO.CL,ICOLCAP.CL,PFCIBEST.CL
Date,,,,
2026-11-02,2780.0,4895.0,24460.5,74480.0
2026-11-03,2695.0,4845.0,24617.0,75820.0
2026-11-04,2750.0,5100.0,24720.5,78020.0
2026-11-05,2520.0,4860.0,24502.5,74900.0
2026-11-06,2720.0,4990.0,24389.5,75460.0


## 3.1 Revisión de calidad de los datos
Antes de calcular, revisamos los retornos diarios más extremos. Encontramos un problema en la fuente: Yahoo Finance reporta para `PFCIBEST.CL` un cierre de 34.900 el 19-feb-2025, entre 42.000 (18-feb) y 44.300 (20-feb), es decir −16,9 % y luego +26,9 %. Según los reportes de prensa bursátil, ese día la acción prácticamente no se movió. Ecopetrol muestra el mismo patrón en esas fechas.

**Regla de limpieza:** si un día tiene un retorno extremo (más de 15 %) y el retorno acumulado de ese día y el siguiente es casi cero (menos de 10 %), el precio de ese día se considera un error y se reemplaza por el del día anterior. Un salto real y permanente **no** se toca. Puedes desactivar la limpieza con `LIMPIAR = False`.

In [4]:
UMBRAL = 0.15        # retorno diario considerado extremo (15 %)
TOLERANCIA = 0.10    # si el retorno de 2 días es casi cero, el precio del día intermedio es sospechoso
LIMPIAR = True       # False = usar los datos tal como vienen de Yahoo Finance


def limpiar_saltos(precios, umbral=UMBRAL, tolerancia=TOLERANCIA):
    p = precios.copy()
    r = precios.pct_change()
    detalle = []
    for col in p.columns:
        hoy = r[col]
        sig = hoy.shift(-1)
        mascara = (hoy.abs() > umbral) & (hoy * sig < 0) & (((1 + hoy) * (1 + sig) - 1).abs() < tolerancia)
        for fecha in hoy.index[mascara.fillna(False)]:
            i = p.index.get_loc(fecha)
            detalle.append({"Fecha": fecha.date(), "Activo": col,
                            "Precio reportado": float(precios.iloc[i][col]),
                            "Retorno del día": float(hoy.loc[fecha]),
                            "Retorno siguiente": float(sig.loc[fecha])})
            p.iloc[i, p.columns.get_loc(col)] = p.iloc[i - 1, p.columns.get_loc(col)]
    columnas = ["Fecha", "Activo", "Precio reportado", "Retorno del día", "Retorno siguiente"]
    tabla = pd.DataFrame(detalle, columns=columnas)
    if len(tabla):
        tabla = tabla.sort_values(["Fecha", "Activo"]).reset_index(drop=True)
    return p, tabla


def extremos(p, n=10):
    r = p.pct_change().stack()
    return r.reindex(r.abs().nlargest(n).index).rename("retorno")


print("10 retornos diarios más extremos ANTES de limpiar:")
print(extremos(precios).to_string())

if LIMPIAR:
    precios, saltos = limpiar_saltos(precios)
    print(f"\nPrecios corregidos: {len(saltos)}")
    if len(saltos):
        print(saltos.to_string())
    print("\n10 retornos diarios más extremos DESPUÉS de limpiar:")
    print(extremos(precios).to_string())

10 retornos diarios más extremos ANTES de limpiar:
Date        Ticker      
2022-05-25  EXITO.CL        0.350688
2023-10-17  EXITO.CL        0.330741
2025-02-20  PFCIBEST.CL     0.269341
2022-05-24  EXITO.CL       -0.256392
2025-02-20  ECOPETROL.CL    0.228022
2025-12-04  EXITO.CL        0.222973
2022-02-22  EXITO.CL        0.207185
2025-12-03  EXITO.CL       -0.192140
2025-02-19  ECOPETROL.CL   -0.183857
            PFCIBEST.CL    -0.169048

Precios corregidos: 4
        Fecha        Activo  Precio reportado  Retorno del día  Retorno siguiente
0  2022-05-24      EXITO.CL       3082.211182        -0.256392           0.350688
1  2025-02-19  ECOPETROL.CL       1556.786499        -0.183857           0.228022
2  2025-02-19   PFCIBEST.CL      29540.900391        -0.169048           0.269341
3  2025-12-03      EXITO.CL       3617.551270        -0.192140           0.222973

10 retornos diarios más extremos DESPUÉS de limpiar:
Date        Ticker      
2023-10-17  EXITO.CL        0.330741
2022-

## 4. Funciones de cálculo
| Indicador | Fórmula |
|---|---|
| Retorno diario | `P_t / P_(t-1) − 1` |
| Retorno anualizado (CAGR) | `(P_final / P_inicial)^(1/años) − 1` |
| Volatilidad anual | `σ_diaria × √252` |
| Sharpe | `(μ_anual − r_f) / σ_anual` |
| Drawdown | `P_t / máximo_previo − 1` |
| Beta | `Cov(activo, mercado) / Var(mercado)` |

In [5]:
def retornos(p):
    return p.pct_change().dropna(how="all")

def retorno_acumulado(p):
    return p.iloc[-1] / p.iloc[0] - 1

def retorno_anualizado(p):
    anios = (p.index[-1] - p.index[0]).days / 365.25
    return (p.iloc[-1] / p.iloc[0]) ** (1 / anios) - 1

def volatilidad_anual(r):
    return r.std() * np.sqrt(DIAS_ANIO)

def sharpe(r, rf=0.0):
    return (r.mean() * DIAS_ANIO - rf) / volatilidad_anual(r)

def drawdown(p):
    return p / p.cummax() - 1

def beta(r, bench):
    return r.apply(lambda s: s.cov(r[bench])) / r[bench].var()

def base_100(p):
    return p / p.iloc[0] * 100

rets = retornos(precios)
print("Retornos calculados:", rets.shape)

Retornos calculados: (1287, 4)


## 5. Indicadores por activo

In [6]:
tabla = pd.DataFrame({
    "Retorno acumulado": retorno_acumulado(precios),
    "Retorno anualizado": retorno_anualizado(precios),
    "Volatilidad anual": volatilidad_anual(rets),
    "Sharpe": sharpe(rets, RF),
    "Máx. drawdown": drawdown(precios).min(),
    f"Beta vs {BENCHMARK}": beta(rets, BENCHMARK),
})

formato = {c: "{:.2%}" for c in ["Retorno acumulado", "Retorno anualizado", "Volatilidad anual", "Máx. drawdown"]}
formato.update({"Sharpe": "{:.2f}", f"Beta vs {BENCHMARK}": "{:.2f}"})
tabla.style.format(formato)

,Retorno acumulado,Retorno anualizado,Volatilidad anual,Sharpe,Máx. drawdown,Beta vs ICOLCAP.CL
Ticker,,,,,,
ECOPETROL.CL,86.77%,13.33%,35.51%,0.41,-37.99%,0.81
EXITO.CL,47.80%,8.14%,38.03%,0.28,-66.59%,0.40
ICOLCAP.CL,120.94%,17.21%,18.00%,0.73,-30.22%,1.00
PFCIBEST.CL,273.55%,30.22%,28.81%,0.90,-32.58%,0.92


## 6. Visualizaciones
### 6.1 Mapa riesgo-retorno
Responde: ¿qué activo dio más retorno por unidad de riesgo? Lo ideal está arriba a la izquierda.

In [7]:
dfp = tabla.reset_index()
dfp.columns = ["Activo"] + list(dfp.columns[1:])
fig = px.scatter(dfp, x="Volatilidad anual", y="Retorno anualizado", text="Activo",
                 color="Sharpe", color_continuous_scale="RdYlGn", title="Riesgo vs. retorno")
fig.update_traces(marker=dict(size=16), textposition="top center")
fig.update_layout(xaxis_tickformat=".0%", yaxis_tickformat=".0%", height=450)
fig.show()

### 6.2 Crecimiento de una inversión de 100
Responde: ¿cuánto habría crecido una inversión inicial y en qué momentos cayó?

In [8]:
fig = px.line(base_100(precios), title="Inversión de 100 (base 100)")
fig.update_layout(yaxis_title="Valor", xaxis_title="", legend_title="", height=450)
fig.show()

### 6.3 Tendencia con medias móviles
Elige el activo y las ventanas en la primera línea.

In [9]:
ACTIVO, CORTA, LARGA = precios.columns[0], 20, 100

s = precios[ACTIVO]
fig = go.Figure()
fig.add_trace(go.Scatter(x=s.index, y=s, name="Precio"))
fig.add_trace(go.Scatter(x=s.index, y=s.rolling(CORTA).mean(), name=f"MA {CORTA}"))
fig.add_trace(go.Scatter(x=s.index, y=s.rolling(LARGA).mean(), name=f"MA {LARGA}"))
fig.update_layout(title=f"{ACTIVO}: precio y medias móviles", height=400)
fig.show()

### 6.4 Drawdown
Responde: ¿qué tan profundas y duraderas fueron las caídas desde un máximo?

In [10]:
fig = px.line(drawdown(precios), title="Drawdown (caída desde el máximo)")
fig.update_layout(yaxis_tickformat=".0%", xaxis_title="", yaxis_title="", legend_title="", height=400)
fig.show()

### 6.5 Volatilidad móvil
Responde: ¿el riesgo cambió con el tiempo?

In [11]:
vol_movil = rets.rolling(VENTANA).std() * np.sqrt(DIAS_ANIO)
fig = px.line(vol_movil.dropna(), title=f"Volatilidad anualizada móvil ({VENTANA} días)")
fig.update_layout(yaxis_tickformat=".0%", xaxis_title="", yaxis_title="", legend_title="", height=400)
fig.show()

### 6.6 Distribución de retornos diarios

In [12]:
largo = rets.melt(var_name="Activo", value_name="Retorno")
fig = px.histogram(largo, x="Retorno", color="Activo", nbins=80, barmode="overlay",
                   opacity=0.6, title="Distribución de retornos diarios")
fig.update_layout(xaxis_tickformat=".0%", height=400)
fig.show()

### 6.7 Correlación
Responde: ¿qué activos ayudan a diversificar? Correlación baja o negativa = mejor diversificación.

In [13]:
corr = rets.corr()
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Correlación de retornos diarios")
fig.update_layout(height=450)
fig.show()

## 7. Lectura automática de resultados

In [14]:
mejor_ret = tabla["Retorno anualizado"].idxmax()
mejor_sh = tabla["Sharpe"].idxmax()
mas_vol = tabla["Volatilidad anual"].idxmax()
menos_vol = tabla["Volatilidad anual"].idxmin()
peor_dd = tabla["Máx. drawdown"].idxmin()

pares = [(a, b, corr.loc[a, b]) for i, a in enumerate(corr.columns) for b in corr.columns[i + 1:]]
a, b, c = min(pares, key=lambda p: p[2])

print(f"• Mayor retorno anualizado: {mejor_ret} ({tabla.loc[mejor_ret, 'Retorno anualizado']:.1%})")
print(f"• Mejor Sharpe (retorno por unidad de riesgo): {mejor_sh} ({tabla.loc[mejor_sh, 'Sharpe']:.2f})")
print(f"• Más volátil: {mas_vol} ({tabla.loc[mas_vol, 'Volatilidad anual']:.1%}) | menos volátil: {menos_vol} ({tabla.loc[menos_vol, 'Volatilidad anual']:.1%})")
print(f"• Mayor caída desde un máximo: {peor_dd} ({tabla.loc[peor_dd, 'Máx. drawdown']:.1%})")
print(f"• Mejor par diversificador (menor correlación): {a} y {b} ({c:.2f})")

• Mayor retorno anualizado: PFCIBEST.CL (30.2%)
• Mejor Sharpe (retorno por unidad de riesgo): PFCIBEST.CL (0.90)
• Más volátil: EXITO.CL (38.0%) | menos volátil: ICOLCAP.CL (18.0%)
• Mayor caída desde un máximo: EXITO.CL (-66.6%)
• Mejor par diversificador (menor correlación): ECOPETROL.CL y EXITO.CL (0.14)


## 8. Interpretación y conclusiones *(completar con los resultados reales)*
1. **Evidencia:** [gráfico o tabla de arriba]
2. **Qué muestra:** [comportamiento observado]
3. **Interpretación financiera:** [significado respecto a la pregunta principal]
4. **Limitaciones:** el análisis es histórico y depende del periodo elegido; solo usa fechas comunes; Yahoo Finance es una fuente gratuita no oficial; si hay activos en distintas monedas, la comparación se distorsiona.

**Conclusión:** [respuesta sustentada a la pregunta principal]

---
### Fuentes de consulta
[yfinance](https://ranaroussi.github.io/yfinance/) · [pandas](https://pandas.pydata.org/docs/) · [Libro del curso](https://github.com/vyolete/proyecto_analitica_financiera)

## 9. Exportar resultados (opcional)

In [15]:
tabla.to_csv("indicadores.csv")
print("Guardado: indicadores.csv")

Guardado: indicadores.csv
